# searching for the best model to train on c4 dataset for asc prediction 
- training loads of models and will hpc the best

# imports and setup

In [ ]:
# Searching for the Best Model for Autism Prediction on C4 Dataset
# Comprehensive ML Model Training and Feature Engineering

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("=== Autism Prediction Model Training ===")
print("Dataset: C4 Matched Balanced (90,539 samples, 55 features)")
print("Target: autism_target (binary classification)")

# 2. data loading and exploration 

In [ ]:
# Load the C4 dataset
c4_data = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv')

print(f"Dataset shape: {c4_data.shape}")
print(f"Target distribution: {c4_data['autism_target'].value_counts()}")

# Basic exploration
print(f"\nFeature types:")
print(f"Numeric features: {len(c4_data.select_dtypes(include=[np.number]).columns)}")
print(f"Categorical features: {len(c4_data.select_dtypes(include=['object']).columns)}")

# Check for missing values
missing_values = c4_data.isnull().sum()
print(f"\nMissing values: {missing_values[missing_values > 0].sum()}")

# Display feature names
print(f"\nAvailable features: {list(c4_data.columns)}")

# FE (existing features)

In [ ]:
# Create a copy for feature engineering
df = c4_data.copy()

print("Creating features from dl_domain_adaption.ipynb...")

def create_aggregate_features(df, prefix, n_items):
    """Create aggregate features for questionnaire items"""
    item_cols = [f"{prefix}_{i}" for i in range(1, n_items+1) if f"{prefix}_{i}" in df.columns]
    if item_cols:
        df[f"{prefix}_total"] = df[item_cols].sum(axis=1)
    return df

# Create questionnaire totals
for prefix, n_items in [('eq', 10), ('aq', 10), ('sqr', 10), ('spq', 10)]:
    df = create_aggregate_features(df, prefix, n_items)

# D-score (Empathy - Social Responsiveness)
if 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['d_score'] = df['eq_total'] - df['sqr_total']

# Age interactions
if 'age' in df.columns and 'aq_total' in df.columns:
    df['age_x_aq'] = df['age'] * df['aq_total']
if 'age' in df.columns and 'eq_total' in df.columns:
    df['age_x_eq'] = df['age'] * df['eq_total']

# Trait interactions
if 'aq_total' in df.columns and 'eq_total' in df.columns:
    df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']

# Ratios for cognitive profiles
if 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)
if 'aq_total' in df.columns and 'eq_total' in df.columns:
    df['aq_eq_ratio'] = df['aq_total'] / (df['eq_total'] + 1e-8)

# Log transformations
if 'aq_total' in df.columns:
    df['log_aq_total'] = np.log1p(np.clip(df['aq_total'], a_min=0, a_max=None))
if 'age' in df.columns:
    df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))

# High trait flags
if 'aq_total' in df.columns:
    df['high_aq'] = (df['aq_total'] > 32).astype(int)
if 'eq_total' in df.columns:
    df['low_eq'] = (df['eq_total'] < 30).astype(int)

print(f"Features after basic engineering: {len(df.columns)}")

# FE (scientific literature features)

In [ ]:
print("Creating additional features based on scientific literature...")

# Age-related features (developmental considerations)
if 'age' in df.columns:
    df['age_squared'] = df['age'] ** 2
    df['age_cubed'] = df['age'] ** 3
    df['log_age'] = np.log1p(df['age'])
    
    # Age groups for developmental stages
    df['age_group'] = pd.cut(df['age'], bins=[0, 18, 25, 35, 50, 100], 
                             labels=['adolescent', 'young_adult', 'adult', 'middle_age', 'senior'])

# Sex-related features (autism prevalence differs by sex)
if 'sex_num' in df.columns:
    # Sex-specific autism patterns
    df['sex_x_aq'] = df['sex_num'] * df['aq_total']
    df['sex_x_eq'] = df['sex_num'] * df['eq_total']
    df['sex_x_sqr'] = df['sex_num'] * df['sqr_total']

# Questionnaire subdomain features (based on factor analysis literature)
if all(f'aq_{i}' in df.columns for i in range(1, 11)):
    # AQ subdomains (Baron-Cohen et al., 2001)
    df['aq_social_skills'] = df[['aq_1', 'aq_7', 'aq_8', 'aq_9', 'aq_10']].sum(axis=1)
    df['aq_attention_switching'] = df[['aq_2', 'aq_4', 'aq_6']].sum(axis=1)
    df['aq_attention_detail'] = df[['aq_3', 'aq_5']].sum(axis=1)

if all(f'eq_{i}' in df.columns for i in range(1, 11)):
    # EQ subdomains (Baron-Cohen & Wheelwright, 2004)
    df['eq_cognitive'] = df[['eq_1', 'eq_3', 'eq_5', 'eq_7', 'eq_9']].sum(axis=1)
    df['eq_affective'] = df[['eq_2', 'eq_4', 'eq_6', 'eq_8', 'eq_10']].sum(axis=1)

if all(f'sqr_{i}' in df.columns for i in range(1, 11)):
    # SQR subdomains (Constantino & Gruber, 2005)
    df['sqr_social_awareness'] = df[['sqr_1', 'sqr_2', 'sqr_3']].sum(axis=1)
    df['sqr_social_cognition'] = df[['sqr_4', 'sqr_5', 'sqr_6']].sum(axis=1)
    df['sqr_social_communication'] = df[['sqr_7', 'sqr_8', 'sqr_9']].sum(axis=1)
    df['sqr_social_motivation'] = df[['sqr_10']].sum(axis=1)

# SPQ subdomain features (Raine, 1991)
if all(f'spq_{i}' in df.columns for i in range(1, 11)):
    df['spq_cognitive_perceptual'] = df[['spq_1', 'spq_2', 'spq_3', 'spq_4']].sum(axis=1)
    df['spq_interpersonal'] = df[['spq_5', 'spq_6', 'spq_7', 'spq_8']].sum(axis=1)
    df['spq_disorganized'] = df[['spq_9', 'spq_10']].sum(axis=1)

print(f"Features after scientific features: {len(df.columns)}")

# 5. FE (statistical features - could be overkill)

In [ ]:
print("Creating statistical features...")

# Z-scores for questionnaire totals
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df.columns:
        df[f'{col}_zscore'] = (df[col] - df[col].mean()) / df[col].std()

# Percentile ranks
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df.columns:
        df[f'{col}_percentile'] = df[col].rank(pct=True)

# Extreme value indicators
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df.columns:
        df[f'{col}_extreme_high'] = (df[col] > df[col].quantile(0.95)).astype(int)
        df[f'{col}_extreme_low'] = (df[col] < df[col].quantile(0.05)).astype(int)

# Three-way interactions
if all(col in df.columns for col in ['aq_total', 'eq_total', 'sqr_total']):
    df['aq_eq_sqr_interaction'] = df['aq_total'] * df['eq_total'] * df['sqr_total']

# Quadratic terms
for col in ['aq_total', 'eq_total', 'sqr_total']:
    if col in df.columns:
        df[f'{col}_squared'] = df[col] ** 2

# Cross-ratio features
if all(col in df.columns for col in ['aq_total', 'eq_total', 'sqr_total']):
    df['aq_eq_cross_ratio'] = df['aq_total'] / (df['eq_total'] + 1e-8)
    df['aq_sqr_cross_ratio'] = df['aq_total'] / (df['sqr_total'] + 1e-8)
    df['eq_sqr_cross_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)

print(f"Features after statistical features: {len(df.columns)}")

# 6. feature engineering (clinical features - again could be overkill)

In [ ]:
print("Creating clinical features...")

# Autism screening thresholds (based on literature)
if 'aq_total' in df.columns:
    df['aq_above_threshold'] = (df['aq_total'] > 26).astype(int)  # Baron-Cohen et al., 2001
    df['aq_high_threshold'] = (df['aq_total'] > 32).astype(int)   # Clinical threshold

# Empathy deficits
if 'eq_total' in df.columns:
    df['eq_below_threshold'] = (df['eq_total'] < 30).astype(int)  # Baron-Cohen & Wheelwright, 2004

# Social responsiveness deficits
if 'sqr_total' in df.columns:
    df['sqr_above_threshold'] = (df['sqr_total'] > 60).astype(int)  # Constantino & Gruber, 2005

# ============================================================================
# DIAGNOSIS-BASED CLINICAL FEATURES
# ============================================================================

# Check for diagnosis columns (I to Q or similar pattern)
diagnosis_cols = [col for col in df.columns if any(diagnosis in col.lower() for diagnosis in 
                   ['adhd', 'autism', 'bipolar', 'depression', 'learning', 'ocd', 'schizophrenia', 'diagnosed'])]

print(f"Found diagnosis columns: {diagnosis_cols}")

if diagnosis_cols:
    # Create diagnosis-based features
    for col in diagnosis_cols:
        if col in df.columns:
            # Binary diagnosis indicators
            df[f'{col}_binary'] = (df[col] > 0).astype(int)
            
            # Diagnosis interactions with questionnaire scores
            if 'aq_total' in df.columns:
                df[f'{col}_x_aq'] = df[col] * df['aq_total']
            if 'eq_total' in df.columns:
                df[f'{col}_x_eq'] = df[col] * df['eq_total']
            if 'sqr_total' in df.columns:
                df[f'{col}_x_sqr'] = df[col] * df['sqr_total']

    # Comorbidity features (based on autism literature)
    if len(diagnosis_cols) > 1:
        # Count of comorbid conditions
        diagnosis_binary_cols = [f'{col}_binary' for col in diagnosis_cols if f'{col}_binary' in df.columns]
        if diagnosis_binary_cols:
            df['comorbidity_count'] = df[diagnosis_binary_cols].sum(axis=1)
            
            # Specific comorbidity patterns (based on literature)
            if all(col in df.columns for col in ['adhd', 'autism']):
                df['autism_adhd_comorbidity'] = ((df['adhd'] > 0) & (df['autism'] > 0)).astype(int)
            
            if all(col in df.columns for col in ['autism', 'depression']):
                df['autism_depression_comorbidity'] = ((df['autism'] > 0) & (df['depression'] > 0)).astype(int)
            
            if all(col in df.columns for col in ['autism', 'ocd']):
                df['autism_ocd_comorbidity'] = ((df['autism'] > 0) & (df['ocd'] > 0)).astype(int)
            
            if all(col in df.columns for col in ['autism', 'learning']):
                df['autism_learning_comorbidity'] = ((df['autism'] > 0) & (df['learning'] > 0)).astype(int)

    # Autism-specific features
    autism_cols = [col for col in diagnosis_cols if 'autism' in col.lower()]
    if autism_cols:
        for col in autism_cols:
            if col in df.columns:
                # Autism severity indicators
                df[f'{col}_severity'] = df[col]  # Assuming higher values = more severe
                
                # Autism with/without other conditions
                if 'comorbidity_count' in df.columns:
                    df[f'{col}_with_comorbidity'] = ((df[col] > 0) & (df['comorbidity_count'] > 1)).astype(int)
                    df[f'{col}_without_comorbidity'] = ((df[col] > 0) & (df['comorbidity_count'] == 1)).astype(int)

# ============================================================================
# OCCUPATION AND DEMOGRAPHIC FEATURES
# ============================================================================

# Occupation features
if 'is_stem_occupation' in df.columns:
    df['stem_x_aq'] = df['is_stem_occupation'] * df['aq_total']
    df['stem_x_eq'] = df['is_stem_occupation'] * df['eq_total']

# Sex category features (one-hot encoded)
sex_cols = [col for col in df.columns if col.startswith('sex_')]
if sex_cols:
    for col in sex_cols:
        if col in df.columns:
            df[f'{col}_x_aq'] = df[col] * df['aq_total']

print(f"Features after clinical features: {len(df.columns)}")

# 7. feature selection adn data prep

In [ ]:
print("Feature selection and preparation...")

# Remove target variable FIRST
X = df.drop(columns=['autism_target'])
y = df['autism_target']

# CRITICAL: Remove ALL autism-related features that could leak
autism_leakage_features = [col for col in X.columns if any(term in col.lower() for term in 
                          ['autism', 'risk_score', 'target'])]
if autism_leakage_features:
    print(f"Removing autism leakage features: {autism_leakage_features}")
    X = X.drop(columns=autism_leakage_features)

# Remove non-numeric columns
X = X.select_dtypes(include=[np.number])

# Remove constant features
constant_features = [col for col in X.columns if X[col].nunique() == 1]
X = X.drop(columns=constant_features)

# Final leakage check
print(f"Final feature set: {X.shape[1]} features")
print(f"Target variable: {y.name}")
print(f"Sample features: {list(X.columns[:10])}")

# CRITICAL: Verify no leakage
print(f"Target in features: {'autism_target' in X.columns}")
print(f"Any 'autism' in features: {any('autism' in col.lower() for col in X.columns)}")
print(f"Any 'risk' in features: {any('risk' in col.lower() for col in X.columns)}")
print(f"Any 'target' in features: {any('target' in col.lower() for col in X.columns)}")

# Handle missing values
X = X.fillna(X.mean())

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# 7.5 checking for leakage before model definitions and training

In [ ]:
# Check for any features that might leak target information
suspicious_features = []
for col in X.columns:
    if any(term in col.lower() for term in ['autism', 'target', 'diagnosis', 'risk']):
        suspicious_features.append(col)

if suspicious_features:
    print(f"WARNING: Suspicious features found: {suspicious_features}")
    X = X.drop(columns=suspicious_features)

# Verify target variable is separate
print(f"Target variable: {c4_data['autism_target'].name}")
print(f"Target distribution: {c4_data['autism_target'].value_counts()}")

# Check for any obvious leakage
if 'autism_target' in c4_data.columns:
    print("✓ Target variable exists")
    # Remove it from features immediately
    feature_data = c4_data.drop(columns=['autism_target'])
    print(f"✓ Target removed from features")



# 8. model definitions

In [ ]:
print("Defining optimized models for autism prediction...")

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=6, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=50, max_depth=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
}

# 9. model training and evals 

In [ ]:
print("Training and evaluating optimized models...")

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    try:
        # Train model
        model.fit(X_train_scaled, y_train)
        
        # Predictions
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred_proba)
        f1 = f1_score(y_test, y_pred)
        
        # Cross-validation (reduced to 3-fold for speed)
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='accuracy', n_jobs=-1)
        cv_f1_scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='f1', n_jobs=-1)
        
        results[name] = {
            'accuracy': accuracy,
            'auc': auc,
            'f1': f1,
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'cv_f1_mean': cv_f1_scores.mean(),
            'cv_f1_std': cv_f1_scores.std(),
            'model': model
        }
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  AUC: {auc:.4f}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        print(f"  CV F1: {cv_f1_scores.mean():.4f} (+/- {cv_f1_scores.std() * 2:.4f})")
        
    except Exception as e:
        print(f"  Error: {e}")
        results[name] = {'error': str(e)}

# 10. results summary 

In [ ]:
print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)

# Create results dataframe
results_df = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': result.get('accuracy', np.nan),
        'AUC': result.get('auc', np.nan),
        'CV_Accuracy': result.get('cv_mean', np.nan),
        'CV_Std': result.get('cv_std', np.nan)
    }
    for name, result in results.items() if 'error' not in result
]).sort_values('AUC', ascending=False)

print(results_df.to_string(index=False))

# Find best model
best_model_name = results_df.iloc[0]['Model']
best_model = results[best_model_name]['model']

print(f"\nBEST MODEL: {best_model_name}")
print(f"Best AUC: {results_df.iloc[0]['AUC']:.4f}")
print(f"Best Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")

# 11. feature importance analysis

In [ ]:
print("Feature importance analysis...")

if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Top 20 most important features:")
    print(feature_importance.head(20).to_string(index=False))
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    plt.barh(range(20), feature_importance['importance'][:20])
    plt.yticks(range(20), feature_importance['feature'][:20])
    plt.xlabel('Feature Importance')
    plt.title(f'Top 20 Features - {best_model_name}')
    plt.tight_layout()
    plt.show()

# 12. detailed analysis of best model 

In [ ]:
print("Detailed analysis of best model...")

# Confusion matrix
y_pred_best = best_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

print("\n" + "="*60)
print("FEATURE ENGINEERING SUMMARY")
print("="*60)
print(f"Original features: 55")
print(f"Engineered features: {X.shape[1]}")
print(f"Total features: {X.shape[1]}")
print(f"Best model: {best_model_name}")
print(f"Best AUC: {results_df.iloc[0]['AUC']:.4f}")
print("="*60)

# phase 2 model performance analysis 

# feature analysis & data understanding 

In [ ]:
# Analyze the age dominance - why is age so important?
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(X['age'][y==0], alpha=0.7, label='Non-autism', bins=20)
plt.hist(X['age'][y==1], alpha=0.7, label='Autism', bins=20)
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend()
plt.title('Age Distribution by Diagnosis')

plt.subplot(1, 2, 2)
age_autism_rate = df.groupby(pd.cut(df['age'], bins=10))['autism_target'].mean()
plt.plot(age_autism_rate.index.astype(str), age_autism_rate.values)
plt.xlabel('Age Group')
plt.ylabel('Autism Rate')
plt.title('Autism Rate by Age Group')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# error analysis 

In [ ]:
# Analyze misclassifications
y_pred_lightgbm = best_model.predict(X_test_scaled)
misclassified = X_test[y_pred_lightgbm != y_test]

print("Misclassification Analysis:")
print(f"Total misclassified: {len(misclassified)}")
print(f"Misclassification rate: {len(misclassified)/len(X_test):.3f}")

# Analyze feature patterns in misclassifications
misclassified_means = misclassified.mean()
correct_means = X_test[y_pred_lightgbm == y_test].mean()
feature_diff = (misclassified_means - correct_means).abs().sort_values(ascending=False)
print("\nTop features that differ in misclassifications:")
print(feature_diff.head(10))

# cross-val stability check 

In [ ]:
# More robust CV to ensure stability
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=cv, scoring='f1')
print(f"10-fold CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# 4. feature selection experiment

In [ ]:
# Test with top features only
top_features = feature_importance.head(30)['feature'].tolist()
X_top = X[top_features]

# Retrain with top features
X_train_top, X_test_top, y_train, y_test = train_test_split(X_top, y, test_size=0.2, random_state=42)
scaler_top = StandardScaler()
X_train_top_scaled = scaler_top.fit_transform(X_train_top)
X_test_top_scaled = scaler_top.transform(X_test_top)

lightgbm_top = lgb.LGBMClassifier(n_estimators=50, random_state=42)
lightgbm_top.fit(X_train_top_scaled, y_train)
y_pred_top = lightgbm_top.predict(X_test_top_scaled)
print(f"Top 30 features performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_top):.4f}")
print(f"AUC: {roc_auc_score(y_test, lightgbm_top.predict_proba(X_test_top_scaled)[:, 1]):.4f}")

# 5. age stratified analysis

In [ ]:
# Test if age-specific models perform better
age_groups = pd.cut(X['age'], bins=[0, 18, 25, 35, 50, 100], labels=['0-18', '19-25', '26-35', '36-50', '50+'])

for age_group in age_groups.unique():
    if pd.isna(age_group):
        continue
    mask = age_groups == age_group
    if mask.sum() > 1000:  # Only if enough samples
        X_age = X[mask]
        y_age = y[mask]
        
        X_train_age, X_test_age, y_train_age, y_test_age = train_test_split(
            X_age, y_age, test_size=0.2, random_state=42, stratify=y_age
        )
        
        scaler_age = StandardScaler()
        X_train_age_scaled = scaler_age.fit_transform(X_train_age)
        X_test_age_scaled = scaler_age.transform(X_test_age)
        
        lightgbm_age = lgb.LGBMClassifier(n_estimators=50, random_state=42)
        lightgbm_age.fit(X_train_age_scaled, y_train_age)
        y_pred_age = lightgbm_age.predict(X_test_age_scaled)
        
        print(f"\nAge group {age_group}:")
        print(f"  Samples: {len(X_age)}")
        print(f"  Accuracy: {accuracy_score(y_test_age, y_pred_age):.4f}")
        print(f"  F1: {f1_score(y_test_age, y_pred_age):.4f}")

# 6. baseline comparison 

In [ ]:
# Compare to simple baselines
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='stratified', random_state=42)
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)
print(f"Dummy classifier F1: {f1_score(y_test, y_pred_dummy):.4f}")

# Majority class baseline
majority = DummyClassifier(strategy='most_frequent', random_state=42)
majority.fit(X_train_scaled, y_train)
y_pred_majority = majority.predict(X_test_scaled)
print(f"Majority class F1: {f1_score(y_test, y_pred_majority):.4f}")

# phase 3 - investigating age distribution 
- age is the biggest predictor which is not great, this might not be representative of population and model is relying on age over autism traits
- age stratified models showed promise

In [ ]:
# understand the age bias
print("=== Age Distribution Analysis ===")
age_autism = df.groupby('autism_target')['age'].describe()
print(age_autism)

# Check for age bias in your dataset
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
df[df['autism_target']==0]['age'].hist(bins=20, alpha=0.7, label='Non-autism')
df[df['autism_target']==1]['age'].hist(bins=20, alpha=0.7, label='Autism')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend()
plt.title('Age Distribution by Diagnosis')

plt.subplot(1, 2, 2)
age_bins = pd.cut(df['age'], bins=10)
autism_rate = df.groupby(age_bins)['autism_target'].mean()
plt.plot(range(len(autism_rate)), autism_rate.values)
plt.xlabel('Age Group')
plt.ylabel('Autism Rate')
plt.title('Autism Rate by Age Group')
plt.xticks(range(len(autism_rate)), [str(x) for x in autism_rate.index], rotation=45)
plt.tight_layout()
plt.show()

# removing age bias

In [ ]:
# Create age-balanced dataset or remove age features
# Option A: Remove age features
X_no_age = X.drop(columns=[col for col in X.columns if 'age' in col.lower()])

# Option B: Age-stratified modeling
# Train separate models for each age group

# Option C: Age-balanced sampling
from sklearn.utils import resample

# Balance age distribution within each diagnosis group
df_balanced = df.copy()
for diagnosis in [0, 1]:
    diagnosis_data = df[df['autism_target'] == diagnosis]
    # Resample to match age distribution of other group
    # Implementation depends on your age distribution analysis

# focusing on questionnaire features 

In [ ]:
# Remove age features and retrain
X_questionnaire = X.drop(columns=[col for col in X.columns if 'age' in col.lower()])
print(f"Features without age: {X_questionnaire.shape[1]}")

# Retrain LightGBM with questionnaire features only
X_train_q, X_test_q, y_train, y_test = train_test_split(
    X_questionnaire, y, test_size=0.2, random_state=42, stratify=y
)

scaler_q = StandardScaler()
X_train_q_scaled = scaler_q.fit_transform(X_train_q)
X_test_q_scaled = scaler_q.transform(X_test_q)

lightgbm_q = lgb.LGBMClassifier(n_estimators=50, random_state=42)
lightgbm_q.fit(X_train_q_scaled, y_train)
y_pred_q = lightgbm_q.predict(X_test_q_scaled)

print(f"Questionnaire-only model:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_q):.4f}")
print(f"  F1: {f1_score(y_test, y_pred_q):.4f}")
print(f"  AUC: {roc_auc_score(y_test, lightgbm_q.predict_proba(X_test_q_scaled)[:, 1]):.4f}")

# feature importance analysis without age 

In [ ]:
# Analyze feature importance without age bias
feature_importance_no_age = pd.DataFrame({
    'feature': X_questionnaire.columns,
    'importance': lightgbm_q.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 features WITHOUT age bias:")
print(feature_importance_no_age.head(20))

# Plot feature importance
plt.figure(figsize=(12, 8))
plt.barh(range(20), feature_importance_no_age['importance'][:20])
plt.yticks(range(20), feature_importance_no_age['feature'][:20])
plt.xlabel('Feature Importance')
plt.title('Top 20 Features - Questionnaire Only (No Age Bias)')
plt.tight_layout()
plt.show()

# compare model performance 

In [ ]:
# Compare with and without age features
print("=== Model Comparison ===")
print("With age features:")
print(f"  Accuracy: 0.6871")
print(f"  F1: 0.6845")
print(f"  AUC: 0.7639")

print("\nWithout age features:")
print(f"  Accuracy: 0.6649")
print(f"  F1: 0.6438")
print(f"  AUC: 0.7277")

print(f"\nPerformance drop:")
print(f"  Accuracy: {((0.6871 - 0.6649) / 0.6871) * 100:.1f}%")
print(f"  F1: {((0.6845 - 0.6438) / 0.6845) * 100:.1f}%")
print(f"  AUC: {((0.7639 - 0.7277) / 0.7639) * 100:.1f}%")

# age stratefied analysis (with age features)

In [ ]:
# Test questionnaire-only model across age groups
age_groups = pd.cut(df['age'], bins=[0, 18, 25, 35, 50, 100], labels=['0-18', '19-25', '26-35', '36-50', '50+'])

for age_group in age_groups.unique():
    if pd.isna(age_group):
        continue
    mask = age_groups == age_group
    if mask.sum() > 1000:
        X_age = X_questionnaire[mask]
        y_age = y[mask]
        
        X_train_age, X_test_age, y_train_age, y_test_age = train_test_split(
            X_age, y_age, test_size=0.2, random_state=42, stratify=y_age
        )
        
        scaler_age = StandardScaler()
        X_train_age_scaled = scaler_age.fit_transform(X_train_age)
        X_test_age_scaled = scaler_age.transform(X_test_age)
        
        lightgbm_age = lgb.LGBMClassifier(n_estimators=50, random_state=42)
        lightgbm_age.fit(X_train_age_scaled, y_train_age)
        y_pred_age = lightgbm_age.predict(X_test_age_scaled)
        
        print(f"\nAge group {age_group} (Questionnaire-only):")
        print(f"  Samples: {len(X_age)}")
        print(f"  Accuracy: {accuracy_score(y_test_age, y_pred_age):.4f}")
        print(f"  F1: {f1_score(y_test_age, y_pred_age):.4f}")

# clinical validation 

In [ ]:
# Analyze which questionnaire features are most important
top_features = feature_importance_no_age.head(10)['feature'].tolist()
print("\nMost important questionnaire features:")
for i, feature in enumerate(top_features, 1):
    print(f"{i}. {feature}")

# Check if these align with clinical autism criteria
clinical_features = ['aq_social_skills', 'eq_cognitive', 'sqr_social_awareness', 'aq_attention_switching']
print(f"\nClinical feature importance:")
for feature in clinical_features:
    if feature in feature_importance_no_age['feature'].values:
        importance = feature_importance_no_age[feature_importance_no_age['feature'] == feature]['importance'].iloc[0]
        print(f"  {feature}: {importance:.1f}")

# creating age matched balanced dataset
- age bias is because the original balanced dataset that was created by 01_explore_C4_data.ipynb made the age groups imbalanced
- 50+: 30% autism rate (3248 vs 7231), 36-50: 66% autism rate (11718 vs 6082), 19-25: 53% autism rate (14182 vs 12378), 26-35: 40% autism rate (4746 vs 6981), 0-18: 29% autism rate (1101 vs 2730)
- recreating the balanced dataset to prevent the age bias 

# 1. load OG data

In [ ]:
# Load the original C4 data (758k samples)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load original data
df_original = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_processed.csv')

print(f"Original dataset shape: {df_original.shape}")
print(f"Original autism rate: {df_original['autism_target'].mean():.3f}")

# Create age-matched balanced dataset
def create_age_matched_balanced_dataset(df, age_bins=10):
    """
    Create a balanced dataset where autism and non-autism groups are matched by age
    """
    # Create age bins
    df['age_bin'] = pd.cut(df['age'], bins=age_bins, labels=False)
    
    balanced_samples = []
    
    for age_bin in df['age_bin'].unique():
        if pd.isna(age_bin):
            continue
            
        # Get samples for this age bin
        age_data = df[df['age_bin'] == age_bin]
        autism_cases = age_data[age_data['autism_target'] == 1]
        control_cases = age_data[age_data['autism_target'] == 0]
        
        print(f"Age bin {age_bin}: Autism={len(autism_cases)}, Control={len(control_cases)}")
        
        # Sample equal numbers from each group
        min_samples = min(len(autism_cases), len(control_cases))
        
        if min_samples > 0:
            autism_sampled = autism_cases.sample(n=min_samples, random_state=42)
            control_sampled = control_cases.sample(n=min_samples, random_state=42)
            
            balanced_samples.append(autism_sampled)
            balanced_samples.append(control_sampled)
    
    # Combine all balanced samples
    df_balanced = pd.concat(balanced_samples, ignore_index=True)
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle
    
    return df_balanced

# Create age-matched balanced dataset
df_age_matched = create_age_matched_balanced_dataset(df_original, age_bins=20)

print(f"\nAge-matched balanced dataset shape: {df_age_matched.shape}")
print(f"Age-matched autism rate: {df_age_matched['autism_target'].mean():.3f}")

# Verify age distribution is balanced
print("\nAge distribution by autism status:")
print(df_age_matched.groupby('autism_target')['age'].describe())

# Save the new balanced dataset
df_age_matched.to_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_age_matched_balanced.csv', index=False)
print("\nSaved age-matched balanced dataset")

# 2. verify age balance 

In [ ]:
# Verify age balance in the new dataset
df_new = pd.read_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_age_matched_balanced.csv')

# Check age distribution by autism status
print("=== AGE DISTRIBUTION VERIFICATION ===")
print(df_new.groupby('autism_target')['age'].describe())

# Check autism rate by age groups
age_groups = pd.cut(df_new['age'], bins=[0, 18, 25, 35, 50, 100], labels=['0-18', '19-25', '26-35', '36-50', '50+'])

print("\n=== AUTISM RATE BY AGE GROUP ===")
for age_group in age_groups.unique():
    if pd.isna(age_group):
        continue
    mask = age_groups == age_group
    autism_rate = df_new[mask]['autism_target'].mean()
    total_samples = mask.sum()
    print(f"{age_group}: {autism_rate:.3f} ({total_samples} samples)")

# retrain model on age matched data 
- gotta re do all FE too

# 1. load and explore age matched dataset

In [ ]:
# Load the age-matched dataset
df_age_matched = pd.read_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_age_matched_balanced.csv')

print("=== AGE-MATCHED DATASET ANALYSIS ===")
print(f"Dataset shape: {df_age_matched.shape}")
print(f"Autism rate: {df_age_matched['autism_target'].mean():.3f}")

# Check existing features
print(f"\nExisting features: {len(df_age_matched.columns)}")
print("Sample features:", list(df_age_matched.columns[:10]))

# Verify age balance
print("\nAge distribution by autism status:")
print(df_age_matched.groupby('autism_target')['age'].describe())

# Check for missing engineered features
expected_features = ['aq_social_skills', 'eq_cognitive', 'sqr_social_cognition', 'd_score', 'age_x_aq']
missing_features = [f for f in expected_features if f not in df_age_matched.columns]
print(f"\nMissing engineered features: {missing_features}")

# 2. complete feature engineering

In [ ]:
print("=== FEATURE ENGINEERING ON AGE-MATCHED DATA ===")

# Feature Engineering - Part 1 (Existing Features)
def create_aggregate_features(df):
    """Create aggregate features from questionnaire items"""
    # AQ subdomain scores
    df['aq_social_skills'] = df[['aq_1', 'aq_2', 'aq_4']].sum(axis=1)
    df['aq_attention_switching'] = df[['aq_3', 'aq_5', 'aq_6']].sum(axis=1)
    df['aq_attention_to_detail'] = df[['aq_7', 'aq_8', 'aq_9', 'aq_10']].sum(axis=1)
    
    # EQ subdomain scores
    df['eq_cognitive'] = df[['eq_1', 'eq_2', 'eq_3', 'eq_4', 'eq_5']].sum(axis=1)
    df['eq_affective'] = df[['eq_6', 'eq_7', 'eq_8', 'eq_9', 'eq_10']].sum(axis=1)
    
    # SQR subdomain scores
    df['sqr_social_awareness'] = df[['sqr_1', 'sqr_2']].sum(axis=1)
    df['sqr_social_cognition'] = df[['sqr_3', 'sqr_4', 'sqr_5']].sum(axis=1)
    df['sqr_social_communication'] = df[['sqr_6', 'sqr_7', 'sqr_8']].sum(axis=1)
    df['sqr_social_motivation'] = df[['sqr_9', 'sqr_10']].sum(axis=1)
    
    # SPQ subdomain scores
    df['spq_cognitive_perceptual'] = df[['spq_1', 'spq_2', 'spq_3', 'spq_4']].sum(axis=1)
    df['spq_interpersonal'] = df[['spq_5', 'spq_6', 'spq_7', 'spq_8']].sum(axis=1)
    df['spq_disorganized'] = df[['spq_9', 'spq_10']].sum(axis=1)
    
    return df

# Apply feature engineering
df_age_matched = create_aggregate_features(df_age_matched)

# Feature Engineering - Part 2 (Scientific Literature-Based Features)
# Age-related features
df_age_matched['age_squared'] = df_age_matched['age'] ** 2
df_age_matched['age_cubed'] = df_age_matched['age'] ** 3
df_age_matched['age_log'] = np.log1p(df_age_matched['age'])

# Age groups
df_age_matched['age_group'] = pd.cut(df_age_matched['age'], 
                                    bins=[0, 18, 25, 35, 50, 100], 
                                    labels=['0-18', '19-25', '26-35', '36-50', '50+'])

# Sex interactions
if 'sex_num' in df_age_matched.columns:
    df_age_matched['sex_x_aq'] = df_age_matched['sex_num'] * df_age_matched['aq_total']
    df_age_matched['sex_x_eq'] = df_age_matched['sex_num'] * df_age_matched['eq_total']
    df_age_matched['sex_x_sqr'] = df_age_matched['sex_num'] * df_age_matched['sqr_total']
    df_age_matched['sex_x_spq'] = df_age_matched['sex_num'] * df_age_matched['spq_total']

# STEM interactions
if 'is_stem_occupation' in df_age_matched.columns:
    df_age_matched['stem_x_aq'] = df_age_matched['is_stem_occupation'] * df_age_matched['aq_total']
    df_age_matched['stem_x_eq'] = df_age_matched['is_stem_occupation'] * df_age_matched['eq_total']

# Feature Engineering - Part 3 (Statistical Features)
# D-score and interactions
df_age_matched['d_score'] = df_age_matched['eq_total'] - df_age_matched['sqr_total']
df_age_matched['age_x_aq'] = df_age_matched['age'] * df_age_matched['aq_total']
df_age_matched['age_x_eq'] = df_age_matched['age'] * df_age_matched['eq_total']
df_age_matched['aq_eq_interaction'] = df_age_matched['aq_total'] * df_age_matched['eq_total']

# Ratios
df_age_matched['eq_sqr_ratio'] = df_age_matched['eq_total'] / (df_age_matched['sqr_total'] + 1e-8)
df_age_matched['aq_eq_ratio'] = df_age_matched['aq_total'] / (df_age_matched['eq_total'] + 1e-8)
df_age_matched['aq_spq_ratio'] = df_age_matched['aq_total'] / (df_age_matched['spq_total'] + 1e-8)

# Transformations
df_age_matched['log_aq_total'] = np.log1p(df_age_matched['aq_total'])
df_age_matched['sqrt_age'] = np.sqrt(df_age_matched['age'])

# High/low trait flags
aq_mean = df_age_matched['aq_total'].mean()
aq_std = df_age_matched['aq_total'].std()
df_age_matched['high_aq'] = (df_age_matched['aq_total'] > aq_mean + aq_std).astype(int)

eq_mean = df_age_matched['eq_total'].mean()
eq_std = df_age_matched['eq_total'].std()
df_age_matched['low_eq'] = (df_age_matched['eq_total'] < eq_mean - eq_std).astype(int)

# Z-scores and percentile ranks
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df_age_matched.columns:
        df_age_matched[f'{col}_zscore'] = (df_age_matched[col] - df_age_matched[col].mean()) / df_age_matched[col].std()
        df_age_matched[f'{col}_percentile'] = df_age_matched[col].rank(pct=True)

# Extreme value indicators
for col in ['aq_total', 'eq_total', 'sqr_total', 'spq_total']:
    if col in df_age_matched.columns:
        q95 = df_age_matched[col].quantile(0.95)
        q05 = df_age_matched[col].quantile(0.05)
        df_age_matched[f'{col}_high_extreme'] = (df_age_matched[col] > q95).astype(int)
        df_age_matched[f'{col}_low_extreme'] = (df_age_matched[col] < q05).astype(int)

# Three-way interactions
df_age_matched['age_sex_aq'] = df_age_matched['age'] * df_age_matched['sex_num'] * df_age_matched['aq_total']
df_age_matched['age_sex_eq'] = df_age_matched['age'] * df_age_matched['sex_num'] * df_age_matched['eq_total']

# Quadratic terms
df_age_matched['aq_total_squared'] = df_age_matched['aq_total'] ** 2
df_age_matched['eq_total_squared'] = df_age_matched['eq_total'] ** 2
df_age_matched['sqr_total_squared'] = df_age_matched['sqr_total'] ** 2

# Cross-ratios
df_age_matched['aq_eq_cross_ratio'] = df_age_matched['aq_total'] * df_age_matched['eq_total'] / (df_age_matched['sqr_total'] + 1e-8)
df_age_matched['aq_sqr_cross_ratio'] = df_age_matched['aq_total'] * df_age_matched['sqr_total'] / (df_age_matched['eq_total'] + 1e-8)
df_age_matched['eq_sqr_cross_ratio'] = df_age_matched['eq_total'] * df_age_matched['sqr_total'] / (df_age_matched['aq_total'] + 1e-8)

print(f"After feature engineering: {df_age_matched.shape}")
print(f"New features created: {df_age_matched.shape[1] - 68}")  # 68 was original count

# 3. feature selection and data prep

In [ ]:
print("=== FEATURE SELECTION AND DATA PREPARATION ===")

# Prepare target and features
y = df_age_matched['autism_target']
X = df_age_matched.drop(['autism_target'], axis=1)

# Remove any features containing 'autism', 'risk_score', or 'target'
leakage_features = [col for col in X.columns if any(term in col.lower() for term in ['autism', 'risk_score', 'target'])]
X = X.drop(columns=leakage_features)

print(f"Removed leakage features: {leakage_features}")

# Remove non-numeric and constant features
X = X.select_dtypes(include=[np.number])
constant_features = X.columns[X.std() == 0]
X = X.drop(columns=constant_features)

print(f"Removed constant features: {len(constant_features)}")

# Handle missing values
missing_counts = X.isnull().sum()
if missing_counts.sum() > 0:
    print(f"Missing values found: {missing_counts.sum()}")
    X = X.fillna(X.mean())
else:
    print("No missing values found")

print(f"Final feature set: {X.shape[1]} features")
print("Sample features:", list(X.columns[:10]))

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Target distribution - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")

# 4. model training and eval

In [ ]:
print("=== TRAINING ALL MODELS ON AGE-MATCHED DATA ===")

# Define optimized models (same as your existing code)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=6, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=50, max_depth=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
}

results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_probs = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_probs)
    f1 = f1_score(y_test, y_pred)
    
    # Cross-validation
    cv_accuracy = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='accuracy', n_jobs=-1)
    cv_f1 = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='f1', n_jobs=-1)
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  AUC: {auc:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  CV Accuracy: {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std()*2:.4f})")
    print(f"  CV F1: {cv_f1.mean():.4f} (+/- {cv_f1.std()*2:.4f})")
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'AUC': auc,
        'F1': f1,
        'CV_Accuracy': cv_accuracy.mean(),
        'CV_Std': cv_accuracy.std()
    })

# Results summary
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1', ascending=False)

print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY (AGE-MATCHED DATA)")
print("="*60)
print(results_df.to_string(index=False))

print(f"\nBEST MODEL: {results_df.iloc[0]['Model']}")
print(f"Best F1: {results_df.iloc[0]['F1']:.4f}")
print(f"Best AUC: {results_df.iloc[0]['AUC']:.4f}")

# 5. feature importance analysis

In [ ]:
print("=== FEATURE IMPORTANCE ANALYSIS ===")

# Get best model
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

print(f"Analyzing feature importance for: {best_model_name}")

# Check if model has feature_importances_ attribute
if hasattr(best_model, 'feature_importances_'):
    print("✓ Model has feature importances")
    
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\nTop 20 features ({best_model_name}):")
    print(feature_importance.head(20).to_string(index=False))
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    plt.barh(range(20), feature_importance['importance'][:20])
    plt.yticks(range(20), feature_importance['feature'][:20])
    plt.xlabel('Feature Importance')
    plt.title(f'Top 20 Features - {best_model_name} (Age-Matched Data)')
    plt.tight_layout()
    plt.show()
    
    # Check if age is still dominant
    age_features = [col for col in feature_importance['feature'][:20] if 'age' in col.lower()]
    print(f"\nAge-related features in top 20: {len(age_features)}")
    if age_features:
        print("Age features:", age_features)
    
    # Check questionnaire features
    questionnaire_features = [col for col in feature_importance['feature'][:20] 
                           if any(q in col.lower() for q in ['aq_', 'eq_', 'sqr_', 'spq_'])]
    print(f"Questionnaire features in top 20: {len(questionnaire_features)}")
    print("Top questionnaire features:", questionnaire_features[:5])
    
    # Detailed analysis
    print(f"\n=== DETAILED FEATURE ANALYSIS ===")
    print(f"Total features: {len(feature_importance)}")
    print(f"Top 5 features:")
    for i, (_, row) in enumerate(feature_importance.head(5).iterrows(), 1):
        print(f"  {i}. {row['feature']}: {row['importance']:.3f}")
    
    # Feature type breakdown
    feature_types = {
        'Age-related': [col for col in feature_importance['feature'] if 'age' in col.lower()],
        'AQ-related': [col for col in feature_importance['feature'] if 'aq' in col.lower()],
        'EQ-related': [col for col in feature_importance['feature'] if 'eq' in col.lower()],
        'SQR-related': [col for col in feature_importance['feature'] if 'sqr' in col.lower()],
        'SPQ-related': [col for col in feature_importance['feature'] if 'spq' in col.lower()],
        'Sex-related': [col for col in feature_importance['feature'] if 'sex' in col.lower()],
        'Other': [col for col in feature_importance['feature'] 
                 if not any(term in col.lower() for term in ['age', 'aq', 'eq', 'sqr', 'spq', 'sex'])]
    }
    
    print(f"\nFeature type breakdown (top 20):")
    for feature_type, features in feature_types.items():
        top_20_features = [f for f in features if f in feature_importance['feature'][:20]]
        if top_20_features:
            print(f"  {feature_type}: {len(top_20_features)} features")
            print(f"    Examples: {top_20_features[:3]}")
    
else:
    print("✗ Model does not have feature importances")
    
    # For models without feature_importances_ (like Logistic Regression)
    if hasattr(best_model, 'coef_'):
        print("Using coefficients for Logistic Regression")
        coef_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': np.abs(best_model.coef_[0])
        }).sort_values('importance', ascending=False)
        
        print(f"\nTop 20 features ({best_model_name} - coefficients):")
        print(coef_importance.head(20).to_string(index=False))
        
        # Plot coefficient importance
        plt.figure(figsize=(12, 8))
        plt.barh(range(20), coef_importance['importance'][:20])
        plt.yticks(range(20), coef_importance['feature'][:20])
        plt.xlabel('|Coefficient|')
        plt.title(f'Top 20 Features - {best_model_name} (Age-Matched Data)')
        plt.tight_layout()
        plt.show()
        
        # Check if age is still dominant
        age_features = [col for col in coef_importance['feature'][:20] if 'age' in col.lower()]
        print(f"\nAge-related features in top 20: {len(age_features)}")
        if age_features:
            print("Age features:", age_features)
        
        # Check questionnaire features
        questionnaire_features = [col for col in coef_importance['feature'][:20] 
                               if any(q in col.lower() for q in ['aq_', 'eq_', 'sqr_', 'spq_'])]
        print(f"Questionnaire features in top 20: {len(questionnaire_features)}")
        print("Top questionnaire features:", questionnaire_features[:5])
    
    else:
        print("Model has no feature importance or coefficient method")

# Compare with original biased model
print(f"\n=== COMPARISON WITH ORIGINAL BIASED MODEL ===")
print("Original model top features (from your previous results):")
print("  1. age (importance: 194)")
print("  2. eq_cognitive (importance: 65)")
print("  3. aq_social_skills (importance: 64)")

print(f"\nNew age-matched model top features:")
if hasattr(best_model, 'feature_importances_'):
    top_features = feature_importance.head(3)
    for i, (_, row) in enumerate(top_features.iterrows(), 1):
        print(f"  {i}. {row['feature']} (importance: {row['importance']:.1f})")
elif hasattr(best_model, 'coef_'):
    top_features = coef_importance.head(3)
    for i, (_, row) in enumerate(top_features.iterrows(), 1):
        print(f"  {i}. {row['feature']} (coefficient: {row['importance']:.1f})")

# Check if age is still in top features
if hasattr(best_model, 'feature_importances_'):
    top_5_features = feature_importance['feature'][:5].tolist()
elif hasattr(best_model, 'coef_'):
    top_5_features = coef_importance['feature'][:5].tolist()
else:
    top_5_features = []

age_in_top_5 = any('age' in f.lower() for f in top_5_features)
print(f"\nAge in top 5 features: {age_in_top_5}")
if not age_in_top_5:
    print("✓ SUCCESS: Age bias has been eliminated!")
else:
    print("⚠ Age is still prominent - may need further investigation")

In [ ]:
# Verify the age-matching actually worked
print("=== VERIFYING AGE-MATCHING ===")

# Load the age-matched dataset
df_age_matched = pd.read_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_age_matched_balanced.csv')

# Check age distribution by autism status
print("Age distribution by autism status:")
print(df_age_matched.groupby('autism_target')['age'].describe())

# Check age correlation with autism_target
age_corr = df_age_matched['age'].corr(df_age_matched['autism_target'])
print(f"\nAge correlation with autism_target: {age_corr:.4f}")

# Check age distribution by age groups
age_groups = pd.cut(df_age_matched['age'], bins=[0, 18, 25, 35, 50, 100], labels=['0-18', '19-25', '26-35', '36-50', '50+'])

print("\nAutism rate by age group:")
for age_group in age_groups.unique():
    if pd.isna(age_group):
        continue
    mask = age_groups == age_group
    autism_rate = df_age_matched[mask]['autism_target'].mean()
    total_samples = mask.sum()
    print(f"{age_group}: {autism_rate:.3f} ({total_samples} samples)")

# Check if there are still age differences
autism_age_mean = df_age_matched[df_age_matched['autism_target'] == 1]['age'].mean()
control_age_mean = df_age_matched[df_age_matched['autism_target'] == 0]['age'].mean()
age_diff = abs(autism_age_mean - control_age_mean)

print(f"\nAge difference between groups: {age_diff:.2f} years")
if age_diff > 1.0:
    print("⚠ WARNING: Age matching may not have worked properly!")
else:
    print("✓ Age matching appears successful")

In [ ]:
# Check if we're creating too many age features
age_features = [col for col in X.columns if 'age' in col.lower()]
print(f"\nAge-related features created: {len(age_features)}")
print("Age features:", age_features)

# Check age feature correlations
age_correlations = X[age_features].corrwith(y)
print(f"\nAge feature correlations with target:")
for feature in age_features:
    corr = age_correlations[feature]
    print(f"  {feature}: {corr:.4f}")

In [ ]:
print("=== TESTING WITHOUT AGE FEATURES ===")

# Remove all age-related features
age_features = [col for col in X.columns if 'age' in col.lower()]
X_no_age = X.drop(columns=age_features)

print(f"Removed {len(age_features)} age-related features")
print(f"Remaining features: {X_no_age.shape[1]}")

# Split and scale
X_train_no_age, X_test_no_age, y_train_no_age, y_test_no_age = train_test_split(
    X_no_age, y, test_size=0.2, random_state=42, stratify=y
)

scaler_no_age = StandardScaler()
X_train_no_age_scaled = scaler_no_age.fit_transform(X_train_no_age)
X_test_no_age_scaled = scaler_no_age.transform(X_test_no_age)

# Test best model without age features
best_model_no_age = LogisticRegression(max_iter=1000, random_state=42)
best_model_no_age.fit(X_train_no_age_scaled, y_train_no_age)

y_pred_no_age = best_model_no_age.predict(X_test_no_age_scaled)
y_probs_no_age = best_model_no_age.predict_proba(X_test_no_age_scaled)[:, 1]

accuracy_no_age = accuracy_score(y_test_no_age, y_pred_no_age)
auc_no_age = roc_auc_score(y_test_no_age, y_probs_no_age)
f1_no_age = f1_score(y_test_no_age, y_pred_no_age)

print(f"\nModel performance WITHOUT age features:")
print(f"  Accuracy: {accuracy_no_age:.4f}")
print(f"  AUC: {auc_no_age:.4f}")
print(f"  F1: {f1_no_age:.4f}")

# Compare with age features
print(f"\nPerformance comparison:")
print(f"  With age features: F1={results_df.iloc[0]['F1']:.4f}, AUC={results_df.iloc[0]['AUC']:.4f}")
print(f"  Without age features: F1={f1_no_age:.4f}, AUC={auc_no_age:.4f}")

# Feature importance without age
coef_importance_no_age = pd.DataFrame({
    'feature': X_no_age.columns,
    'importance': np.abs(best_model_no_age.coef_[0])
}).sort_values('importance', ascending=False)

print(f"\nTop 10 features WITHOUT age:")
print(coef_importance_no_age.head(10).to_string(index=False))

In [ ]:
# Check the original age-matching code
print("=== VERIFYING AGE-MATCHING ALGORITHM ===")

# Load original data
df_original = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_processed.csv')

print(f"Original dataset shape: {df_original.shape}")
print(f"Original autism rate: {df_original['autism_target'].mean():.3f}")

# Check age distribution in original data
print("\nOriginal age distribution by autism status:")
print(df_original.groupby('autism_target')['age'].describe())

# The age-matching might need to be more aggressive
def create_strict_age_matched_dataset(df, age_bin_size=5):
    """
    Create age-matched dataset with smaller age bins for better matching
    """
    # Create smaller age bins
    df['age_bin'] = pd.cut(df['age'], bins=np.arange(0, 101, age_bin_size), labels=False)
    
    balanced_samples = []
    
    for age_bin in df['age_bin'].unique():
        if pd.isna(age_bin):
            continue
            
        age_data = df[df['age_bin'] == age_bin]
        autism_cases = age_data[age_data['autism_target'] == 1]
        control_cases = age_data[age_data['autism_target'] == 0]
        
        min_samples = min(len(autism_cases), len(control_cases))
        
        if min_samples > 0:
            autism_sampled = autism_cases.sample(n=min_samples, random_state=42)
            control_sampled = control_cases.sample(n=min_samples, random_state=42)
            
            balanced_samples.append(autism_sampled)
            balanced_samples.append(control_sampled)
    
    df_balanced = pd.concat(balanced_samples, ignore_index=True)
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return df_balanced

# Create stricter age-matched dataset
df_strict_age_matched = create_strict_age_matched_dataset(df_original, age_bin_size=2)

print(f"\nStrict age-matched dataset shape: {df_strict_age_matched.shape}")
print(f"Strict age-matched autism rate: {df_strict_age_matched['autism_target'].mean():.3f}")

# Verify strict age matching
print("\nStrict age-matched distribution:")
print(df_strict_age_matched.groupby('autism_target')['age'].describe())

# Save strict age-matched dataset
df_strict_age_matched.to_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_strict_age_matched_balanced.csv', index=False)
print("\nSaved strict age-matched dataset")

# age features removed

In [ ]:
# Test the strict age-matched dataset
print("=== TESTING STRICT AGE-MATCHED DATASET ===")

# Load strict age-matched dataset
df_strict = pd.read_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/data_c4_strict_age_matched_balanced.csv')

print(f"Strict age-matched dataset shape: {df_strict.shape}")
print(f"Autism rate: {df_strict['autism_target'].mean():.3f}")

# Apply feature engineering (same as before)
df_strict = create_aggregate_features(df_strict)

# Add all the same features as before
# (Copy the feature engineering code from your previous run)

# Prepare data
y_strict = df_strict['autism_target']
X_strict = df_strict.drop(['autism_target'], axis=1)

# Remove leakage and constant features
leakage_features = [col for col in X_strict.columns if any(term in col.lower() for term in ['autism', 'risk_score', 'target'])]
X_strict = X_strict.drop(columns=leakage_features)
X_strict = X_strict.select_dtypes(include=[np.number])
constant_features = X_strict.columns[X_strict.std() == 0]
X_strict = X_strict.drop(columns=constant_features)
X_strict = X_strict.fillna(X_strict.mean())

print(f"Strict dataset features: {X_strict.shape[1]}")

# Split and scale
X_train_strict, X_test_strict, y_train_strict, y_test_strict = train_test_split(
    X_strict, y_strict, test_size=0.2, random_state=42, stratify=y_strict
)

scaler_strict = StandardScaler()
X_train_strict_scaled = scaler_strict.fit_transform(X_train_strict)
X_test_strict_scaled = scaler_strict.transform(X_test_strict)

# Test without age features
age_features_strict = [col for col in X_strict.columns if 'age' in col.lower()]
X_strict_no_age = X_strict.drop(columns=age_features_strict)

X_train_strict_no_age, X_test_strict_no_age, y_train_strict_no_age, y_test_strict_no_age = train_test_split(
    X_strict_no_age, y_strict, test_size=0.2, random_state=42, stratify=y_strict
)

scaler_strict_no_age = StandardScaler()
X_train_strict_no_age_scaled = scaler_strict_no_age.fit_transform(X_train_strict_no_age)
X_test_strict_no_age_scaled = scaler_strict_no_age.transform(X_test_strict_no_age)

# Test LightGBM without age features
lightgbm_strict_no_age = lgb.LGBMClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
lightgbm_strict_no_age.fit(X_train_strict_no_age_scaled, y_train_strict_no_age)

y_pred_strict_no_age = lightgbm_strict_no_age.predict(X_test_strict_no_age_scaled)
y_probs_strict_no_age = lightgbm_strict_no_age.predict_proba(X_test_strict_no_age_scaled)[:, 1]

accuracy_strict_no_age = accuracy_score(y_test_strict_no_age, y_pred_strict_no_age)
auc_strict_no_age = roc_auc_score(y_test_strict_no_age, y_probs_strict_no_age)
f1_strict_no_age = f1_score(y_test_strict_no_age, y_pred_strict_no_age)

print(f"\nStrict age-matched dataset WITHOUT age features:")
print(f"  Accuracy: {accuracy_strict_no_age:.4f}")
print(f"  AUC: {auc_strict_no_age:.4f}")
print(f"  F1: {f1_strict_no_age:.4f}")

# Feature importance without age
feature_importance_strict_no_age = pd.DataFrame({
    'feature': X_strict_no_age.columns,
    'importance': lightgbm_strict_no_age.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 features (strict age-matched, no age):")
print(feature_importance_strict_no_age.head(10).to_string(index=False))

# cross validation stability check 

In [ ]:
print("=== CROSS-VALIDATION STABILITY ANALYSIS ===")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

# Use the final clean dataset
X_final = X_strict_no_age  # Your 65 features without age
y_final = y_strict

# 10-fold cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
f1_scores = []
auc_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_final, y_final)):
    X_train_fold, X_val_fold = X_final.iloc[train_idx], X_final.iloc[val_idx]
    y_train_fold, y_val_fold = y_final.iloc[train_idx], y_final.iloc[val_idx]
    
    # Scale features
    scaler_fold = StandardScaler()
    X_train_fold_scaled = scaler_fold.fit_transform(X_train_fold)
    X_val_fold_scaled = scaler_fold.transform(X_val_fold)
    
    # Train model
    model_fold = lgb.LGBMClassifier(n_estimators=50, max_depth=6, random_state=42)
    model_fold.fit(X_train_fold_scaled, y_train_fold)
    
    # Predict
    y_pred_fold = model_fold.predict(X_val_fold_scaled)
    y_probs_fold = model_fold.predict_proba(X_val_fold_scaled)[:, 1]
    
    # Calculate metrics
    f1_fold = f1_score(y_val_fold, y_pred_fold)
    auc_fold = roc_auc_score(y_val_fold, y_probs_fold)
    
    f1_scores.append(f1_fold)
    auc_scores.append(auc_fold)
    
    print(f"Fold {fold+1}: F1={f1_fold:.4f}, AUC={auc_fold:.4f}")

print(f"\nCross-validation results:")
print(f"F1: {np.mean(f1_scores):.4f} (+/- {np.std(f1_scores)*2:.4f})")
print(f"AUC: {np.mean(auc_scores):.4f} (+/- {np.std(auc_scores)*2:.4f})")

In [ ]:
print("=== FEATURE SELECTION ANALYSIS ===")

# Test with different feature subsets
feature_subsets = {
    'Top 20': 20,
    'Top 30': 30,
    'Top 40': 40,
    'All 65': 65
}

for subset_name, n_features in feature_subsets.items():
    # Get top features
    top_features = feature_importance_strict_no_age.head(n_features)['feature'].tolist()
    X_subset = X_final[top_features]
    
    # Split and scale
    X_train_subset, X_test_subset, y_train_subset, y_test_subset = train_test_split(
        X_subset, y_final, test_size=0.2, random_state=42, stratify=y_final
    )
    
    scaler_subset = StandardScaler()
    X_train_subset_scaled = scaler_subset.fit_transform(X_train_subset)
    X_test_subset_scaled = scaler_subset.transform(X_test_subset)
    
    # Train model
    model_subset = lgb.LGBMClassifier(n_estimators=50, max_depth=6, random_state=42)
    model_subset.fit(X_train_subset_scaled, y_train_subset)
    
    # Predict
    y_pred_subset = model_subset.predict(X_test_subset_scaled)
    y_probs_subset = model_subset.predict_proba(X_test_subset_scaled)[:, 1]
    
    # Calculate metrics
    f1_subset = f1_score(y_test_subset, y_pred_subset)
    auc_subset = roc_auc_score(y_test_subset, y_probs_subset)
    
    print(f"{subset_name} features: F1={f1_subset:.4f}, AUC={auc_subset:.4f}")

In [ ]:
print("=== AGE-STRATIFIED PERFORMANCE (NO AGE FEATURES) ===")

# Add age back just for stratification
df_strict_with_age = df_strict.copy()
age_groups = pd.cut(df_strict_with_age['age'], bins=[0, 18, 25, 35, 50, 100], 
                    labels=['0-18', '19-25', '26-35', '36-50', '50+'])

for age_group in age_groups.unique():
    if pd.isna(age_group):
        continue
    mask = age_groups == age_group
    if mask.sum() > 1000:  # Only test groups with sufficient samples
        X_age_group = X_final[mask]
        y_age_group = y_final[mask]
        
        X_train_age, X_test_age, y_train_age, y_test_age = train_test_split(
            X_age_group, y_age_group, test_size=0.2, random_state=42, stratify=y_age_group
        )
        
        scaler_age = StandardScaler()
        X_train_age_scaled = scaler_age.fit_transform(X_train_age)
        X_test_age_scaled = scaler_age.transform(X_test_age)
        
        model_age = lgb.LGBMClassifier(n_estimators=50, max_depth=6, random_state=42)
        model_age.fit(X_train_age_scaled, y_train_age)
        
        y_pred_age = model_age.predict(X_test_age_scaled)
        y_probs_age = model_age.predict_proba(X_test_age_scaled)[:, 1]
        
        f1_age = f1_score(y_test_age, y_pred_age)
        auc_age = roc_auc_score(y_test_age, y_probs_age)
        
        print(f"{age_group}: F1={f1_age:.4f}, AUC={auc_age:.4f} ({len(X_age_group)} samples)")

In [ ]:
print("=== BASELINE COMPARISON ===")

from sklearn.dummy import DummyClassifier

# Random baseline
dummy_random = DummyClassifier(strategy='uniform', random_state=42)
dummy_random.fit(X_train_strict_no_age_scaled, y_train_strict_no_age)
y_pred_dummy = dummy_random.predict(X_test_strict_no_age_scaled)
f1_dummy = f1_score(y_test_strict_no_age, y_pred_dummy)

# Majority class baseline
dummy_majority = DummyClassifier(strategy='most_frequent', random_state=42)
dummy_majority.fit(X_train_strict_no_age_scaled, y_train_strict_no_age)
y_pred_majority = dummy_majority.predict(X_test_strict_no_age_scaled)
f1_majority = f1_score(y_test_strict_no_age, y_pred_majority)

print(f"Random baseline F1: {f1_dummy:.4f}")
print(f"Majority baseline F1: {f1_majority:.4f}")
print(f"Your model F1: {f1_strict_no_age:.4f}")
print(f"Improvement over random: {((f1_strict_no_age - f1_dummy) / f1_dummy * 100):.1f}%")

In [ ]:
print("=== CLINICAL FEATURE ANALYSIS ===")

# Analyze which clinical domains are most important
clinical_domains = {
    'Social Communication': ['aq_social_skills', 'sqr_social_communication', 'sqr_social_cognition'],
    'Empathy': ['eq_affective', 'eq_cognitive'],
    'Executive Function': ['aq_attention_switching', 'aq_attention_to_detail'],
    'Schizotypal Traits': ['spq_4', 'spq_5', 'spq_cognitive_perceptual', 'spq_interpersonal']
}

for domain, features in clinical_domains.items():
    domain_features = [f for f in features if f in feature_importance_strict_no_age['feature'].values]
    if domain_features:
        domain_importance = feature_importance_strict_no_age[
            feature_importance_strict_no_age['feature'].isin(domain_features)
        ]['importance'].sum()
        print(f"{domain}: {domain_importance:.1f} importance")
        print(f"  Features: {domain_features}")

# quick test of ensemble methods before hpc 

In [ ]:
# Test ensemble methods
from sklearn.ensemble import VotingClassifier

# Create ensemble of top 3 models
estimators = [
    ('lightgbm', lgb.LGBMClassifier(n_estimators=100, random_state=42)),
    ('xgb', xgb.XGBClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

ensemble = VotingClassifier(estimators=estimators, voting='soft')
ensemble.fit(X_train_strict_no_age_scaled, y_train_strict_no_age)

y_pred_ensemble = ensemble.predict(X_test_strict_no_age_scaled)
y_probs_ensemble = ensemble.predict_proba(X_test_strict_no_age_scaled)[:, 1]

f1_ensemble = f1_score(y_test_strict_no_age, y_pred_ensemble)
auc_ensemble = roc_auc_score(y_test_strict_no_age, y_probs_ensemble)

print(f"Ensemble performance: F1={f1_ensemble:.4f}, AUC={auc_ensemble:.4f}")